# Format-5 registration

Every volume in the new store is resampled onto the **same 256 × 256 × 128 mm, 1 mm RAS box**, whose origin is the brain center of a skull-on MNI152 T1 (FSL 1 mm). After this step a given anatomical point should sit at nearly the same voxel in every scan — CT or MRI, any sequence, any head size or gantry tilt. That is the space the 3D encoder’s millimetre RoPE assumes.

Format 4 only **centered** the head in that box. Format 5 still estimates that center (it is the starting translation) and then **registers** the scan to the template:

1. Downsample scan and template to 2 mm. Clamp intensities so air/bone do not dominate (CT: [−300, 700] HU; MRI: 0.5–99.5 percentiles).
2. **Rigid** Mattes mutual information (MI needs no shared intensity scale, so CT and T2 both fit the T1 template).
3. **Affine** refinement about the template brain center (scale + shear on top of rigid).
4. Accept the affine only if it is near rigid-plus-scale — singular values in `[0.65, 1.5]`, rotation ≤ 35°, and MI improved. Otherwise keep rigid; if that also fails, keep the centering translation and set `reg_ok = False`. Nothing is dropped.
5. **One** resample of the native-resolution scan through that transform onto the 1 mm grid. Native units are kept (HU / robust z). A brain mask is written beside the image.

This notebook reads whatever format-5 zips the reprocess has already written (the store is mixed v4/v5 until the shards finish). Re-run the harvest cell as more datasets land.

Use the `ml-torch` kernel. If imports fail: `export PYTHONPATH=/home/roku/brain-model:/home/roku/.local/lib/python3.12/site-packages`.

In [ ]:
import os, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

os.environ.pop("WORLD_SIZE", None)
os.environ.pop("RANK", None)
os.environ.pop("LOCAL_RANK", None)

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
USER_SITE = Path.home() / ".local" / "lib" / "python3.12" / "site-packages"
for p in (ROOT, USER_SITE):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from brain.common.paths import SCRATCH
from brain.common.windowing import PRESETS, asinh_window
from brain.prep import geometry, pipeline, registration, store
from brain.prep.index import folder_of

STORE = SCRATCH / "data" / "processed" / "slices_zarr"
SMOKE = SCRATCH / "data" / "processed" / "slices_zarr_v5_smoke"
TEMPLATE = SCRATCH / "templates" / "MNI152_T1_1mm.nii.gz"
TEMPLATE_BRAIN = SCRATCH / "templates" / "MNI152_T1_1mm_brain_mask.nii.gz"
os.environ["BRAIN_TEMPLATE"] = str(TEMPLATE)

N_PER_DATASET = 1
SEED = 0
# Families we care about even if still format-4 in the store (prep has not rewritten them yet).
WEAK_FOLDERS = ("piop1", "piop2", "id1000", "pixar", "oasis1")
print("store", STORE)
print("template", TEMPLATE.exists(), TEMPLATE_BRAIN.exists())

In [ ]:
tpl = registration.load_template(str(TEMPLATE), str(TEMPLATE_BRAIN))
grid = registration.output_grid(tpl, pipeline.SIZE)
tmpl = geometry.resample(tpl.full, grid, 0.0)           # (Z, Y, X) on the store grid
tmpl_brain = geometry.resample(tpl.brain, grid, 0.0, nearest=True) > 0.5
print("template grid", tmpl.shape, "brain frac", float(tmpl_brain.mean()), "center_mm", tpl.center_mm)

In [ ]:
def manifests(root: Path, pattern: str = "manifest*.jsonl") -> pd.DataFrame:
    files = sorted(root.glob(pattern))
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_json(f, lines=True) for f in files], ignore_index=True)
    df = df.drop_duplicates(["dataset", "case_id"], keep="last")
    df = df.loc[df["status"].eq("ok")].copy()
    if "skipped_existing" in df.columns:
        df = df.loc[~df["skipped_existing"].fillna(0).astype(bool)]
    df["root"] = str(root)
    df["folder"] = df["dataset"].map(folder_of)
    df["zip"] = [store.path_for(root, f, c) for f, c in zip(df["folder"], df["case_id"])]
    df["brain_zip"] = [store.path_for(root, f, c, "brain") for f, c in zip(df["folder"], df["case_id"])]
    df = df.loc[df["zip"].map(Path.exists)].copy()
    return df.reset_index(drop=True)

# Main store: only the in-progress v5 shard logs (old v4 manifests are huge and not this format).
v5 = pd.concat(
    [manifests(STORE, "manifest_v5*.jsonl"), manifests(SMOKE)],
    ignore_index=True,
).drop_duplicates(["dataset", "case_id"], keep="last")
v5 = v5.loc[v5["reg_ok"].notna()].reset_index(drop=True)
if v5.empty:
    raise SystemExit("no format-5 manifests yet — wait for prep-v5 to write a few cases")

print(f"{len(v5)} format-5 volumes across {v5.dataset.nunique()} datasets")
print(v5.groupby("dataset").agg(
    n=("case_id", "count"),
    ok=("reg_ok", "mean"),
    rot=("reg_rot_deg", "median"),
    scale=("reg_scale", "median"),
    brain=("brain_frac", "median"),
).round(3).to_string())

In [ ]:
def _weak_score(g: pd.DataFrame) -> pd.Series:
    fail = (~g["reg_ok"].fillna(False).astype(bool)).astype(float)
    rot = pd.to_numeric(g.get("reg_rot_deg"), errors="coerce").fillna(0)
    scale = (pd.to_numeric(g.get("reg_scale"), errors="coerce").fillna(1) - 1).abs()
    return fail * 100 + rot + 40 * scale


def pick(df: pd.DataFrame, n=N_PER_DATASET, by="family") -> pd.DataFrame:
    """n volumes per family (or dataset). Prefer a rejected fit, else highest rotation/scale."""
    from brain.prep.index import dataset_family
    out = []
    key = df["dataset"].map(dataset_family) if by == "family" else df["dataset"]
    for _, g in df.groupby(key, sort=True):
        out.append(g.assign(_s=_weak_score(g)).sort_values("_s", ascending=False).head(n).drop(columns="_s"))
    return pd.concat(out, ignore_index=True) if out else df.iloc[0:0]


def row_from_zip(path: Path, dataset: str) -> pd.Series:
    attrs = store.read_attrs(path, "image")
    return pd.Series({
        "dataset": dataset, "case_id": path.stem, "zip": str(path),
        "brain_zip": str(path.with_name(path.stem + ".brain.zip")),
        "root": str(path.parent.parent), **attrs,
    })


def load_vol(row) -> tuple[np.ndarray, np.ndarray | None]:
    img = np.asarray(store.read(Path(row["zip"]), "image"), dtype=np.float32).squeeze()
    bp = Path(row["brain_zip"])
    brain = np.asarray(store.read(bp, "mask"), dtype=np.uint8).squeeze() > 0 if bp.is_file() else None
    return img, brain


def _win(vol, kind):
    c, w = (PRESETS["ct"]["soft"] if str(kind).startswith("ct") else PRESETS["mri"]["tissue"])
    return asinh_window(vol, c, w)


def _unit(x):
    lo, hi = np.percentile(x, [1, 99])
    return np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)


def checkerboard(a, b, n=8):
    h, w = a.shape
    out = a.copy()
    bh, bw = max(h // n, 1), max(w // n, 1)
    for i in range(n):
        for j in range(n):
            if (i + j) % 2:
                out[i * bh:(i + 1) * bh, j * bw:(j + 1) * bw] = b[i * bh:(i + 1) * bh, j * bw:(j + 1) * bw]
    return out


def ortho(vol):
    z, y, x = (s // 2 for s in vol.shape)
    return vol[z], vol[:, y, :], np.rot90(vol[:, :, x])


def show_case(row, tmpl=tmpl, figsize=(12, 7)):
    vol, brain = load_vol(row)
    kind = row.get("kind") or row.get("modality") or "mri"
    wv, wt = _unit(_win(vol, kind)), _unit(_win(tmpl, "mri"))
    va, vc, vs = ortho(wv)
    ta, tc, ts = ortho(wt)
    panels = [
        (va, "axial"), (vc, "coronal"), (vs, "sagittal"),
        (checkerboard(va, ta), "axial checker vs template"),
        (checkerboard(vc, tc), "coronal checker"),
        (checkerboard(vs, ts), "sagittal checker"),
    ]
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    for ax, (im, title) in zip(axes.ravel(), panels):
        ax.imshow(im, cmap="gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    if brain is not None:
        bz = brain[brain.shape[0] // 2]
        if bz.any():
            axes[0, 0].contour(bz, levels=[0.5], colors="lime", linewidths=0.8)
    def _f(key, spec):
        v = row.get(key)
        try:
            return format(float(v), spec)
        except (TypeError, ValueError):
            return "nan"
    fig.suptitle(
        f"{row['dataset']} / {row['case_id']}   "
        f"reg={row.get('reg_method')} ok={row.get('reg_ok')}  rot={_f('reg_rot_deg', '.1f')}°  "
        f"scale={_f('reg_scale', '.2f')}  aniso={_f('reg_aniso', '.2f')}  "
        f"brain={row.get('brain_method')} {_f('brain_frac', '.2f')}",
        fontsize=10)
    fig.tight_layout()
    return fig


sample = pick(v5, n=1, by="family")

# PIOP / ID1000 / PIXAR / OASIS: pull from the store even if still format 4 (not in v5 manifests yet).
extra = []
for folder in WEAK_FOLDERS:
    zips = sorted(p for p in (STORE / folder).glob("*.zip")
                  if not any(p.name.endswith(t) for t in (".brain.zip", ".seg.zip", ".ICH.zip", ".lesion.zip")))
    if not zips:
        continue
    in_v5 = sample["dataset"].eq(folder) | sample["case_id"].isin([p.stem for p in zips])
    if folder in set(sample.get("dataset", pd.Series(dtype=str))) or folder in set(v5.get("dataset", pd.Series(dtype=str))):
        g = v5[v5["dataset"] == folder]
        if len(g):
            extra.append(g.assign(_s=_weak_score(g)).sort_values("_s", ascending=False).head(1).drop(columns="_s"))
            continue
    extra.append(row_from_zip(zips[len(zips)//2], folder).to_frame().T)

if extra:
    sample = pd.concat([sample, *extra], ignore_index=True).drop_duplicates(["dataset", "case_id"])

print(sample[["dataset", "case_id", "reg_ok", "reg_method", "reg_rot_deg", "reg_scale", "brain_frac"]].to_string(index=False))

Each case is axial / checker-vs-MNI / sagittal of the stored volume. If registration worked, skull, ventricles and midline should line up in the checker even when one panel is CT (or T2, FLAIR) and the other is the T1 template. Lime contour is the stored brain mask.

`pick(..., by="family")` takes the **weakest** volume per family (rejected fit, else highest rotation). `WEAK_FOLDERS` also loads PIOP / ID1000 / PIXAR / OASIS from the store even if they are still format 4. PNGs from a batch pass live in `notebooks/inspect_out/`.

`reg_ok=False` means affine (and maybe rigid) was rejected; those are the interesting failures.

In [ ]:
for _, row in sample.iterrows():
    show_case(row)
    plt.show()

## One volume, all slices

Set `CASE` to any `dataset/case_id` from the harvest table (or a row you care about). Axial mosaic through the 128-slice slab; anatomy should sit in the middle, not jammed against z=0.

In [ ]:
CASE = None  # e.g. "aisd/0091538" — default: first sample row

if CASE:
    ds, cid = CASE.split("/", 1)
    row = v5[(v5.dataset == ds) & (v5.case_id == cid)].iloc[0]
else:
    row = sample.iloc[0]

fig = show_case(row, figsize=(12, 7))
plt.show()

vol, brain = load_vol(row)
kind = row.get("kind") or row.get("modality") or "mri"
zs = np.linspace(8, vol.shape[0] - 9, 16, dtype=int)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, z in zip(axes.ravel(), zs):
    ax.imshow(_win(vol[z], kind), cmap="gray")
    if brain is not None and brain[z].any():
        ax.contour(brain[z], levels=[0.5], colors="lime", linewidths=0.5)
    ax.set_title(f"z={z}", fontsize=8)
    ax.axis("off")
fig.suptitle(f"{row['dataset']}/{row['case_id']}  axial mosaic", fontsize=10)
fig.tight_layout()
plt.show()